# 2026 COMP90042 Project
*Make sure you change the file name with your group id.*

# Readme
*If there is something to be noted for the marker, please mention here.*

*If you are planning to implement a program with Object Oriented Programming style, please put those the bottom of this ipynb file*

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/COMP90042_2026/data"
OUTPUT_DIR = "/content/drive/MyDrive/COMP90042_2026/outputs"

##1.1 Imports and Configurations

In [ ]:
import json
import re
import string
import unicodedata
from pathlib import Path
from typing import Any


import numpy as np
import pandas as pd

import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer
from nltk.corpus import stopwords

from tqdm.auto import tqdm

#Configurations
DATA_DIR = Path("/content/drive/MyDrive/COMP90042_2026/data")
OUTPUT_DIR = Path("/content/drive/MyDrive/COMP90042_2026/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USE_STEMMING = True
REMOVE_STOPWORDS = False  # Set True for a BM25 stopword-removal ablation
RANDOM_SEED = 42
STEMMER = SnowballStemmer("english")

#Downloads
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
BM25_STOPWORDS = set(stopwords.words("english"))


##1.2 Json read/wrtie utilities

In [ ]:
DATA_DIR = Path(DATA_DIR)
OUTPUT_DIR = Path(OUTPUT_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def load_json(filename: str) -> dict[str, Any]:

    path = DATA_DIR / filename

    if not path.exists():
        raise FileNotFoundError(f"Could not find file: {path}")

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    return data


def write_json(data: dict[str, Any], filename: str) -> None:

    path = OUTPUT_DIR / filename

    with path.open("w", encoding="utf-8") as f:
        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )
        f.write("\n")

    print(f"Saved JSON to: {path}")

##1.2 Data preprocess

In [ ]:
"""
Preprocessing strategy

For sparse retrieval with BM25:
- Unicode normalisation
- Whitespace cleanup
- Lowercasing
- Regex-based tokenisation designed for scientific text
- Keep useful scientific/numeric tokens such as co2, 1.5, 10%, w/m2, sea-level
- Optional stopword removal for retrieval ablations
- Optional Snowball stemming, applied only to purely alphabetic tokens

For dense retrieval / transformer models:
- Unicode normalisation
- Whitespace cleanup only
"""

TOKEN_RE = re.compile(r"[a-zA-Z0-9]+(?:[._/%°+-][a-zA-Z0-9]+)*")
WHITESPACE_RE = re.compile(r"\s+")
CONTROL_RE = re.compile(r"[\u0000-\u0008\u000b\u000c\u000e-\u001f\u007f]")

PROCESSED_DIR = OUTPUT_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

BM25_EVIDENCE_CACHE_PATH = PROCESSED_DIR / (
    f"evidence_bm25_tokens_stem_{USE_STEMMING}_stoprm_{REMOVE_STOPWORDS}.json"
)
TRANSFORMER_EVIDENCE_CACHE_PATH = PROCESSED_DIR / "evidence_transformer_text.json"


def normalize_text(text: str, lowercase: bool = True) -> str:
    text = unicodedata.normalize("NFC", text)
    text = CONTROL_RE.sub(" ", text)
    if lowercase:
        text = text.lower()
    return WHITESPACE_RE.sub(" ", text).strip()


def preprocess_for_bm25(
    text: str,
    use_stemming: bool = USE_STEMMING,
    remove_stopwords: bool = REMOVE_STOPWORDS,
) -> list[str]:
    text = normalize_text(text, lowercase=True)
    tokens = [match.group(0) for match in TOKEN_RE.finditer(text)]
    if remove_stopwords:
        tokens = [token for token in tokens if token not in BM25_STOPWORDS]
    if use_stemming:
        tokens = [STEMMER.stem(token) if token.isalpha() else token for token in tokens]
    return tokens


def preprocess_for_transformer(text: str) -> str:
    return normalize_text(text, lowercase=False)


def save_bm25_tokenized_evidence(
    evidence: dict[str, str],
    output_path: Path,
    use_stemming: bool = USE_STEMMING,
    remove_stopwords: bool = REMOVE_STOPWORDS,
):
    evidence_ids = list(evidence.keys())
    tokenized_evidence = []

    for evidence_id in tqdm(evidence_ids, desc="Tokenizing evidence for BM25"):
        tokenized_evidence.append(
            preprocess_for_bm25(
                evidence[evidence_id],
                use_stemming=use_stemming,
                remove_stopwords=remove_stopwords,
            )
        )

    payload = {
        "use_stemming": use_stemming,
        "remove_stopwords": remove_stopwords,
        "evidence_ids": evidence_ids,
        "tokenized_evidence": tokenized_evidence,
    }
    write_json(payload, output_path.name if output_path.parent == OUTPUT_DIR else output_path.relative_to(OUTPUT_DIR))
    return evidence_ids, tokenized_evidence


def load_bm25_tokenized_evidence(cache_path: Path):
    with cache_path.open("r", encoding="utf-8") as f:
        payload = json.load(f)
    return payload["evidence_ids"], payload["tokenized_evidence"]


def get_or_create_bm25_tokenized_evidence(
    evidence: dict[str, str],
    cache_path: Path,
    use_stemming: bool = USE_STEMMING,
    remove_stopwords: bool = REMOVE_STOPWORDS,
    force_rebuild: bool = False,
):
    if cache_path.exists() and not force_rebuild:
        print(f"Loading BM25 token cache from: {cache_path}")
        return load_bm25_tokenized_evidence(cache_path)

    return save_bm25_tokenized_evidence(
        evidence=evidence,
        output_path=cache_path,
        use_stemming=use_stemming,
        remove_stopwords=remove_stopwords,
    )


def save_transformer_evidence_text(evidence: dict[str, str], output_path: Path):
    payload = {
        evidence_id: preprocess_for_transformer(text)
        for evidence_id, text in tqdm(evidence.items(), desc="Normalizing evidence text")
    }
    write_json(payload, output_path.name if output_path.parent == OUTPUT_DIR else output_path.relative_to(OUTPUT_DIR))
    return payload


def get_or_create_transformer_evidence_text(
    evidence: dict[str, str],
    cache_path: Path,
    force_rebuild: bool = False,
):
    if cache_path.exists() and not force_rebuild:
        print(f"Loading transformer evidence cache from: {cache_path}")
        with cache_path.open("r", encoding="utf-8") as f:
            return json.load(f)
    return save_transformer_evidence_text(evidence, cache_path)


In [ ]:
evidence = load_json("evidence.json")

bm25_evidence_ids, tokenized_evidence = get_or_create_bm25_tokenized_evidence(
    evidence=evidence,
    cache_path=BM25_EVIDENCE_CACHE_PATH,
    use_stemming=USE_STEMMING,
    remove_stopwords=REMOVE_STOPWORDS,
    force_rebuild=False,
)

transformer_evidence = get_or_create_transformer_evidence_text(
    evidence=evidence,
    cache_path=TRANSFORMER_EVIDENCE_CACHE_PATH,
    force_rebuild=False,
)

# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [ ]:
!pip install rank-bm25

In [ ]:
# ============================================================
# BM25 retrieval baseline
# ============================================================
from rank_bm25 import BM25Okapi

train_claims = load_json("train-claims.json")
dev_claims = load_json("dev-claims.json")
test_claims = load_json("test-claims-unlabelled.json")

bm25 = BM25Okapi(tokenized_evidence)


def bm25_retrieve_rank_bm25(claim_text: str, top_k: int = 5):
    """
    Retrieve top-k evidence passages for one claim using rank_bm25.

    Returns:
        list of (evidence_id, score)
    """
    query_tokens = preprocess_for_bm25(
        claim_text,
        use_stemming=USE_STEMMING,
        remove_stopwords=REMOVE_STOPWORDS,
    )
    scores = bm25.get_scores(query_tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [
        (bm25_evidence_ids[idx], float(scores[idx]))
        for idx in top_indices
    ]


def make_bm25_predictions(
    claims: dict[str, dict[str, Any]],
    top_k: int = 5,
    default_label: str = "NOT_ENOUGH_INFO",
):
    predictions = {}
    for claim_id, claim in tqdm(claims.items(), desc="Retrieving evidence"):
        retrieved = bm25_retrieve_rank_bm25(claim["claim_text"], top_k=top_k)
        predictions[claim_id] = {
            "claim_text": claim["claim_text"],
            "claim_label": default_label,
            "evidences": [evidence_id for evidence_id, _ in retrieved],
        }
    return predictions


dev_bm25_predictions = make_bm25_predictions(dev_claims, top_k=5)
bm25_output_name = f"dev-bm25-rank-bm25-stem_{USE_STEMMING}_stoprm_{REMOVE_STOPWORDS}.json"
write_json(dev_bm25_predictions, bm25_output_name)


# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [ ]:
def evaluate_predictions(predictions: dict[str, dict[str, Any]], groundtruth: dict[str, dict[str, Any]]):
    evidence_fscores = []
    label_correct = []

    for claim_id, gold in sorted(groundtruth.items()):
        pred = predictions.get(claim_id, {})
        label_correct.append(float(pred.get("claim_label") == gold["claim_label"]))

        predicted_evidence = pred.get("evidences", [])
        evidence_fscore = 0.0
        if isinstance(predicted_evidence, list) and predicted_evidence:
            predicted_set = set(predicted_evidence)
            correct = sum(1 for evidence_id in gold["evidences"] if evidence_id in predicted_set)
            if correct > 0:
                precision = correct / len(predicted_evidence)
                recall = correct / len(gold["evidences"])
                evidence_fscore = 2 * precision * recall / (precision + recall)
        evidence_fscores.append(evidence_fscore)

    mean_f = float(np.mean(evidence_fscores)) if evidence_fscores else 0.0
    mean_acc = float(np.mean(label_correct)) if label_correct else 0.0
    harmonic = 0.0 if mean_f == 0.0 and mean_acc == 0.0 else 2 * mean_f * mean_acc / (mean_f + mean_acc)

    return {
        "Evidence Retrieval F-score (F)": mean_f,
        "Claim Classification Accuracy (A)": mean_acc,
        "Harmonic Mean of F and A": harmonic,
    }


dev_scores = evaluate_predictions(dev_bm25_predictions, dev_claims)
for metric, value in dev_scores.items():
    print(f"{metric} = {value}")


## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed*